# Stage 1 Validation — Hybrid Tutorial CSV Backfill, Causal Feature Pipeline (`F`) & Feast

Interactive validation notebook for Tickets 01–02b inside Docker (`make fetch-data && make up`) or offline:

- `shared_store` inspects the containerized `feast_data` volume populated by `feast` and `feature` without mutating it.
- `sandbox_store` loads real Hopsworks Tutorial CSVs from `RAW_DATA_DIR` (`data/raw/`) when downloaded (or falls back to `BankingEventSimulator` when offline without `data/raw/`) and runs Pandera, causal `t - 1`, hybrid incremental/drift, and Feast join checks.

In [ ]:
import os
import tempfile
from datetime import UTC, datetime
from pathlib import Path

from banking_fti.adapters import FeastFeatureStore
from banking_fti.core.contracts import RawBankingBatch
from banking_fti.core.validation import SchemaValidationError, validate_raw_batch
from banking_fti.features.causal_aggs import compute_card_window_aggs
from banking_fti.features.csv_loader import (
    generate_incremental_from_backfill,
    has_tutorial_csvs,
    load_tutorial_csv_backfill,
)
from banking_fti.features.pipeline import FeaturePipeline
from banking_fti.features.simulator import BankingEventSimulator, SimulatorConfig

repo_path = Path(os.environ.get("FEAST_REPO_PATH", "feature_repo")).resolve()
shared_default = Path(tempfile.gettempdir()) / "banking_fti_feast_shared"
shared_data_dir = Path(os.environ.get("FEAST_DATA_DIR", str(shared_default))).resolve()
shared_store = FeastFeatureStore(repo_path=repo_path, data_dir=shared_data_dir)

sandbox_default = Path(tempfile.gettempdir()) / "banking_fti_feast_notebook_sandbox"
sandbox_data_dir = Path(
    os.environ.get("FEAST_NOTEBOOK_SANDBOX_DIR", str(sandbox_default))
).resolve()
sandbox_store = FeastFeatureStore(repo_path=repo_path, data_dir=sandbox_data_dir)

raw_dir = Path(os.environ.get("RAW_DATA_DIR", "data/raw")).resolve()
registered_entities = sorted(entity.name for entity in shared_store.list_entities())
registered_views = sorted(view.name for view in shared_store.list_feature_views())
print("Entities:", registered_entities)
print("FeatureViews:", registered_views)
print("Tutorial CSVs available:", has_tutorial_csvs(raw_dir))
print(
    "Shared container credit_applications rows:",
    len(shared_store.read_feature_group("credit_applications")),
)

In [ ]:
t0 = datetime(2026, 2, 1, 0, 0, tzinfo=UTC)
sim = BankingEventSimulator(SimulatorConfig(num_customers=30, backfill_days=30, seed=42))
using_real_csvs = has_tutorial_csvs(raw_dir)
if using_real_csvs:
    backfill = load_tutorial_csv_backfill(raw_dir)
    print(
        "Loaded Hopsworks Tutorial CSV backfill:",
        len(backfill.credit_applications),
        "apps,",
        len(backfill.card_events),
        "tx",
    )
else:
    backfill = sim.generate_backfill(end_time=t0)
    print("Using deterministic simulator fallback (run `make fetch-data` for tutorial CSVs).")

bad_apps = backfill.credit_applications.copy()
bad_apps.loc[0, "dti_ratio"] = -0.25
bad_batch = RawBankingBatch(
    customer_profiles=backfill.customer_profiles,
    card_events=backfill.card_events,
    credit_applications=bad_apps,
    batch_id="invalid-demo",
)

pandera_rejected_invalid = False
try:
    validate_raw_batch(bad_batch)
except SchemaValidationError as exc:
    pandera_rejected_invalid = True
    print("Pandera rejected invalid batch as expected:", exc.dataset_name)

assert pandera_rejected_invalid

In [ ]:
sample_events = backfill.card_events.head(500).copy()
ordered = sample_events.sort_values("event_timestamp").reset_index(drop=True)
target_cc = str(ordered.loc[0, "cc_num"])
future_indices = ordered.index[ordered["cc_num"] == target_cc]
mutated = ordered.copy()
if len(future_indices) > 1:
    mutated.loc[future_indices[-1], "amount"] = 99999.0

baseline_ordered = compute_card_window_aggs(ordered)
mutated_aggs = compute_card_window_aggs(mutated)
first_idx = int(future_indices[0])
causal_exact_match = baseline_ordered.iloc[first_idx].equals(mutated_aggs.iloc[first_idx])
print("Causal t - 1 invariance exact match:", causal_exact_match)
assert causal_exact_match

In [ ]:
pipeline = FeaturePipeline(feature_store=sandbox_store)
backfill_res = pipeline.run_batch(backfill)
if using_real_csvs:
    nominal_inc = generate_incremental_from_backfill(
        backfill, window_hours=24, simulate_drift=False, batch_seq=1
    )
    drift_inc = generate_incremental_from_backfill(
        nominal_inc, window_hours=24, simulate_drift=True, batch_seq=2
    )
else:
    nominal_inc = sim.generate_incremental_batch(
        start_time=t0, window_hours=24, simulate_drift=False, batch_seq=1
    )
    drift_inc = sim.generate_incremental_batch(
        start_time=t0, window_hours=24, simulate_drift=True, batch_seq=2
    )

nominal_res = pipeline.run_batch(
    nominal_inc,
    history_card_events=backfill.card_events,
    reference_stats=backfill_res.group_stats,
)
drift_res = pipeline.run_batch(
    drift_inc,
    history_card_events=backfill.card_events,
    reference_stats=backfill_res.group_stats,
)
drift_detected_on_inc = bool(drift_res.drift_detected)
print("Nominal drift_detected:", nominal_res.drift_detected)
print("Simulated drift_detected:", drift_detected_on_inc)
print("Sandbox Parquet files:", sorted(p.name for p in sandbox_store.data_dir.glob("*.parquet")))

In [ ]:
apps = sandbox_store.read_feature_group("credit_applications")
entity_df = apps.loc[:, ["application_id", "customer_id", "event_timestamp"]].tail(10)

train_df = sandbox_store.get_historical_features(
    entity_df,
    ["customer_profiles", "card_window_aggs", "credit_applications"],
)
batch_df = sandbox_store.get_historical_features(
    entity_df,
    ["customer_profiles", "card_window_aggs", "credit_applications"],
    exclude_labels=True,
)

train_has_label = "default_label" in train_df.columns and not bool(
    train_df["home_ownership"].isna().any()
)
batch_omits_label = "default_label" not in batch_df.columns
print("train_df shape:", train_df.shape, "has_label:", train_has_label)
print("batch_df shape:", batch_df.shape, "omits_label:", batch_omits_label)
train_df.head()